<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/Phase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

print("--- INITIATING WATER MASK GATE ---")

print("Loading Phase 4 Matrix...")
df = pd.read_csv("nalbari_phase4_human_spillover.csv.gz")

# 1. Define the physical constraint
breeding_threshold_mm = 1.0

# 2. Generate the binary boolean mask (1 or 0)
print(f"Generating binary Water Mask (Threshold: {breeding_threshold_mm} mm)...")
df['water_mask'] = (df['net_standing_water'] >= breeding_threshold_mm).astype(int)

# 3. Apply the Spatial AND-Gate
print("Applying physical constraints to Human Spillover Hazard...")
df['final_spillover_hazard'] = df['human_spillover_hazard'] * df['water_mask']

# 4. Export the finalized feature matrix
output_file = "nalbari_phase4_masked_hazard.csv.gz"
print(f"Compressing finalized matrix...")
df.to_csv(output_file, index=False, compression="gzip")

print(f"Gate applied successfully. Triggering download for {output_file}...")
files.download(output_file)

print("\n--- SAMPLE OUTPUT (Water Mask Verification) ---")
# Filter to show a mix of dry and wet days to prove the mask works
sample_view = df[['time', 'hexagon', 'net_standing_water', 'water_mask', 'human_spillover_hazard', 'final_spillover_hazard']]
print("Showing an active rain event transition:")
print(sample_view[(sample_view['net_standing_water'] > 0) | (sample_view['water_mask'] == 0)].sample(15).sort_values(by=['net_standing_water']).head(15))

--- INITIATING WATER MASK GATE ---
Loading Phase 4 Matrix...
Generating binary Water Mask (Threshold: 1.0 mm)...
Applying physical constraints to Human Spillover Hazard...
Compressing finalized matrix...
Gate applied successfully. Triggering download for nalbari_phase4_masked_hazard.csv.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SAMPLE OUTPUT (Water Mask Verification) ---
Showing an active rain event transition:
               time          hexagon  net_standing_water  water_mask  \
2290508  2023-05-13  883ce1d8bdfffff            0.000000           0   
1345113  2022-03-29  883ce1cd01fffff            0.000000           0   
1391852  2021-04-17  883ce1ce03fffff            2.193612           1   
2318258  2021-05-23  883ce1d909fffff          165.259494           1   
350806   2022-02-10  883ce0a5d3fffff          317.714126           1   
2737650  2021-05-30  883ce569a5fffff          381.563162           1   
2097095  2021-06-19  883ce1d667fffff          537.291112           1   
461251   2021-09-13  883ce0ad93fffff          591.885952           1   
866440   2021-10-22  883ce1c2cdfffff          631.508310           1   
2373383  2022-06-02  883ce1d98bfffff          787.369591           1   
2679660  2021-07-14  883ce568dbfffff          840.660563           1   
738658   2022-09-20  883ce1c1a3fffff         1

In [6]:
import pandas as pd
import numpy as np
from google.colab import files

print("--- INITIATING PHASE 5: RISK CLASSIFICATION ENGINE ---")

print("Loading Phase 4 Masked Matrix...")
df = pd.read_csv("nalbari_phase4_masked_hazard.csv.gz")

# 1. Isolate the "Active" hazard scores to calculate realistic statistical thresholds
# We ignore the 0s, otherwise the dry winter months will artificially pull our thresholds down
active_hazards = df[df['final_spillover_hazard'] > 0]['final_spillover_hazard']

print(f"Calculating statistical thresholds across {len(active_hazards):,} active biological events...")
threshold_warning = np.percentile(active_hazards, 75)
threshold_critical = np.percentile(active_hazards, 90)

print(f"  -> Level 2 (Warning) Threshold: {threshold_warning:,.2f}")
print(f"  -> Level 3 (Critical) Threshold: {threshold_critical:,.2f}")

# 2. Apply the Categorical Classification Logic
def classify_risk(hazard):
    if hazard == 0:
        return 0  # Safe
    elif hazard < threshold_warning:
        return 1  # Monitor
    elif hazard < threshold_critical:
        return 2  # Warning
    else:
        return 3  # Critical Action

print("Classifying 3 million spatial-temporal records...")
df['alert_level'] = df['final_spillover_hazard'].apply(classify_risk)

# 3. Final Export
output_file = "nalbari_phase5_final_radar_output.csv.gz"
print("Compressing Final Radar Output...")
df.to_csv(output_file, index=False, compression="gzip")

print(f"Pipeline Complete. Triggering download for {output_file}...")
files.download(output_file)

print("\n--- SYSTEM SUMMARY (Total Days Across All Zones) ---")
print(df['alert_level'].value_counts().sort_index().rename({
    0: 'Level 0 (Safe)',
    1: 'Level 1 (Monitor)',
    2: 'Level 2 (Warning)',
    3: 'Level 3 (Critical)'
}))

--- INITIATING PHASE 5: RISK CLASSIFICATION ENGINE ---
Loading Phase 4 Masked Matrix...
Calculating statistical thresholds across 2,413,579 active biological events...
  -> Level 2 (Warning) Threshold: 113,519.90
  -> Level 3 (Critical) Threshold: 180,993.95
Classifying 3 million spatial-temporal records...
Compressing Final Radar Output...
Pipeline Complete. Triggering download for nalbari_phase5_final_radar_output.csv.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SYSTEM SUMMARY (Total Days Across All Zones) ---
alert_level
Level 0 (Safe)         511166
Level 1 (Monitor)     1810184
Level 2 (Warning)      362037
Level 3 (Critical)     241358
Name: count, dtype: int64


In [8]:
# 1. Restore the Enterprise Geospatial Environment (Required if Colab restarted)
!pip install -q rasterio geopandas rasterstats shapely h3

import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
from rasterstats import zonal_stats
import rasterio
from rasterio.windows import from_bounds
import os
import numpy as np

print("--- INITIATING FINAL TRIAD GATE: GLW4 AMPLIFYING HOST ---")

pig_raster_path = "glw4_pigs.tif"

# The Hard Stop
if not os.path.exists(pig_raster_path):
    raise FileNotFoundError(f"CRITICAL HALT: {pig_raster_path} not found. Ensure it is uploaded to Colab.")

print("Loading Masked Phase 4 Matrix...")
df = pd.read_csv("nalbari_phase4_masked_hazard.csv.gz")

unique_hexagons = df['hexagon'].unique()
print(f"Generating spatial polygons for {len(unique_hexagons)} vector zones...")

hex_polygons = [{"hexagon": hex_id, "geometry": Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(hex_id)])} for hex_id in unique_hexagons]
gdf_hex = gpd.GeoDataFrame(hex_polygons, crs="EPSG:4326")

print("Cropping massive GLW4 Livestock raster to the Nalbari bounding box...")
minx, miny, maxx, maxy = gdf_hex.total_bounds

with rasterio.open(pig_raster_path) as src:
    window = from_bounds(minx, miny, maxx, maxy, src.transform)
    transform = src.window_transform(window)
    cropped_pigs = src.read(1, window=window)
    nodata_value = src.nodata

print("Executing Zonal Summation for Swine Density...")
# Using 'mean' or 'sum' depending on how FAO structured the 10km pixel values.
# For density rasters (pigs per sq km), 'mean' is often safer for downscaling,
# but we will calculate the absolute sum relative to the hexagon area.
pig_stats = zonal_stats(gdf_hex, cropped_pigs, affine=transform, stats="mean", nodata=nodata_value)

# Convert density (pigs/pixel) back to absolute count for the hexagon (Hex area is ~0.73 km^2)
gdf_hex['pig_population'] = [stat['mean'] * 0.73 if stat['mean'] is not None else 0 for stat in pig_stats]

print("Applying the Final Eco-Triad Constraint...")
df_final = pd.merge(df, gdf_hex[['hexagon', 'pig_population']], on='hexagon', how='left')

# THE FINAL MATHEMATICAL GATE
# Logarithmic dampener: 10 pigs trigger the outbreak, 1,000 pigs saturate it.
df_final['pig_multiplier'] = np.log1p(df_final['pig_population'])

df_final['triad_spillover_hazard'] = df_final['final_spillover_hazard'] * df_final['pig_multiplier']

output_file = "nalbari_phase5_true_radar.csv.gz"
df_final.to_csv(output_file, index=False, compression="gzip")

print(f"Eco-Triad Complete. True Biological Radar saved as {output_file}")

from google.colab import files
files.download(output_file)

print("\n--- SAMPLE TRIAD OUTPUT ---")
# Show a random sample where there is standing water and humans, so we can see the pig multiplier in action
sample_view = df_final[(df_final['water_mask'] == 1) & (df_final['human_population'] > 0)]
print(sample_view[['hexagon', 'human_population', 'pig_population', 'final_spillover_hazard', 'triad_spillover_hazard']].drop_duplicates(subset=['hexagon']).head(15))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00
--- INITIATING FINAL TRIAD GATE: GLW4 AMPLIFYING HOST ---
Loading Masked Phase 4 Matrix...
Generating spatial polygons for 2671 vector zones...
Cropping massive GLW4 Livestock raster to the Nalbari bounding box...
Executing Zonal Summation for Swine Density...
Applying the Final Eco-Triad Constraint...
Eco-Triad Complete. True Biological Radar saved as nalbari_phase5_true_radar.csv.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SAMPLE TRIAD OUTPUT ---
               hexagon  human_population  pig_population  \
89     883ce03401fffff        268.382660             0.0   
1184   883ce03403fffff        429.255981             0.0   
2279   883ce03405fffff         84.044037             0.0   
3374   883ce03407fffff        153.178619             0.0   
4469   883ce03409fffff        508.896362             0.0   
5564   883ce0340bfffff        315.111023             0.0   
6659   883ce0340dfffff        526.246460             0.0   
7754   883ce03411fffff        401.907898             0.0   
8849   883ce03413fffff        450.251282             0.0   
9944   883ce03415fffff        480.027435             0.0   
11039  883ce03417fffff        235.106384             0.0   
12134  883ce03419fffff        414.377441             0.0   
13229  883ce0341bfffff        496.478271             0.0   
14324  883ce0341dfffff        309.682343             0.0   
15419  883ce03429fffff        916.333435             0.0   

       fin

In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
from rasterstats import zonal_stats
import rasterio
from rasterio.windows import from_bounds
import os
import numpy as np

print("--- INITIATING ARMORED TRIAD GATE: GLW4 AMPLIFYING HOST ---")

pig_raster_path = "glw4_pigs.tif"

if not os.path.exists(pig_raster_path):
    raise FileNotFoundError(f"CRITICAL HALT: {pig_raster_path} not found.")

print("Loading Masked Phase 4 Matrix...")
df = pd.read_csv("nalbari_phase4_masked_hazard.csv.gz")

unique_hexagons = df['hexagon'].unique()
print(f"Generating spatial polygons for {len(unique_hexagons)} vector zones...")

hex_polygons = [{"hexagon": hex_id, "geometry": Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(hex_id)])} for hex_id in unique_hexagons]
gdf_hex = gpd.GeoDataFrame(hex_polygons, crs="EPSG:4326")

with rasterio.open(pig_raster_path) as src:
    raster_crs = src.crs
    print(f"Raster CRS detected as: {raster_crs}")

    # DEFENSE 1: Reproject hexagons to match the raster's physical reality
    print("Reprojecting vector zones to align with UN spatial coordinates...")
    gdf_hex_proj = gdf_hex.to_crs(raster_crs)

    minx, miny, maxx, maxy = gdf_hex_proj.total_bounds

    print("Cropping massive GLW4 Livestock raster to the true Nalbari bounding box...")
    window = from_bounds(minx, miny, maxx, maxy, src.transform)
    transform = src.window_transform(window)
    cropped_pigs = src.read(1, window=window)
    nodata_value = src.nodata

print("Executing Zonal Summation for Swine Density...")
# DEFENSE 2: all_touched=True ensures tiny hexagons inherit the massive pixel's density
pig_stats = zonal_stats(gdf_hex_proj, cropped_pigs, affine=transform, stats="mean", nodata=nodata_value, all_touched=True)

# Convert density (pigs/pixel) back to absolute count for the hexagon (Hex area is ~0.73 km^2)
gdf_hex['pig_population'] = [stat['mean'] * 0.73 if stat['mean'] is not None else 0 for stat in pig_stats]

print("Applying the Final Eco-Triad Constraint...")
df_final = pd.merge(df, gdf_hex[['hexagon', 'pig_population']], on='hexagon', how='left')

# The Multiplier: Logarithmic dampening
df_final['pig_multiplier'] = np.log1p(df_final['pig_population'])

df_final['triad_spillover_hazard'] = df_final['final_spillover_hazard'] * df_final['pig_multiplier']

output_file = "nalbari_phase5_true_radar.csv.gz"
df_final.to_csv(output_file, index=False, compression="gzip")

print(f"Eco-Triad Complete. True Biological Radar saved as {output_file}")

from google.colab import files
files.download(output_file)

print("\n--- SAMPLE TRIAD OUTPUT (Verification) ---")
sample_view = df_final[(df_final['water_mask'] == 1) & (df_final['human_population'] > 0)]
print(sample_view[['hexagon', 'human_population', 'pig_population', 'pig_multiplier', 'final_spillover_hazard', 'triad_spillover_hazard']].drop_duplicates(subset=['hexagon']).head(15))

--- INITIATING ARMORED TRIAD GATE: GLW4 AMPLIFYING HOST ---
Loading Masked Phase 4 Matrix...
Generating spatial polygons for 2671 vector zones...
Raster CRS detected as: EPSG:4326
Reprojecting vector zones to align with UN spatial coordinates...
Cropping massive GLW4 Livestock raster to the true Nalbari bounding box...
Executing Zonal Summation for Swine Density...
Applying the Final Eco-Triad Constraint...
Eco-Triad Complete. True Biological Radar saved as nalbari_phase5_true_radar.csv.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SAMPLE TRIAD OUTPUT (Verification) ---
               hexagon  human_population  pig_population  pig_multiplier  \
89     883ce03401fffff        268.382660        9.595934        2.360470   
1184   883ce03403fffff        429.255981        9.401568        2.341957   
2279   883ce03405fffff         84.044037        9.595934        2.360470   
3374   883ce03407fffff        153.178619        9.401568        2.341957   
4469   883ce03409fffff        508.896362        9.595934        2.360470   
5564   883ce0340bfffff        315.111023        9.401568        2.341957   
6659   883ce0340dfffff        526.246460        9.790300        2.378648   
7754   883ce03411fffff        401.907898        9.401568        2.341957   
8849   883ce03413fffff        450.251282        9.401568        2.341957   
9944   883ce03415fffff        480.027435        9.401568        2.341957   
11039  883ce03417fffff        235.106384        9.401568        2.341957   
12134  883ce03419fffff        414.377441    

In [10]:
import pandas as pd
import numpy as np
from google.colab import files

print("--- INITIATING FINAL RISK CLASSIFICATION ---")

print("Loading True Eco-Triad Radar...")
df = pd.read_csv("nalbari_phase5_true_radar.csv.gz")

# 1. Isolate the "Active" hazard scores to calculate realistic thresholds
active_hazards = df[df['triad_spillover_hazard'] > 0]['triad_spillover_hazard']

print(f"Calculating statistical thresholds across {len(active_hazards):,} active events...")
threshold_warning = np.percentile(active_hazards, 75)
threshold_critical = np.percentile(active_hazards, 90)

print(f"  -> Level 2 (Warning) Threshold: {threshold_warning:,.2f}")
print(f"  -> Level 3 (Critical) Threshold: {threshold_critical:,.2f}")

# 2. Apply the Categorical Classification Logic
def classify_risk(hazard):
    if hazard == 0:
        return 0  # Safe (Dry, Cold, No Humans, or No Pigs)
    elif hazard < threshold_warning:
        return 1  # Monitor
    elif hazard < threshold_critical:
        return 2  # Warning
    else:
        return 3  # Critical Action

print("Classifying spatial-temporal records against the new Eco-Triad scale...")
df['alert_level'] = df['triad_spillover_hazard'].apply(classify_risk)

# 3. Final Export
output_file = "nalbari_je_radar_final_classified.csv.gz"
print("Compressing Final Radar Output...")
df.to_csv(output_file, index=False, compression="gzip")

print(f"Pipeline Complete. Triggering download for {output_file}...")
files.download(output_file)

print("\n--- SYSTEM SUMMARY (Total Days Across All Zones) ---")
print(df['alert_level'].value_counts().sort_index().rename({
    0: 'Level 0 (Safe)',
    1: 'Level 1 (Monitor)',
    2: 'Level 2 (Warning)',
    3: 'Level 3 (Critical)'
}))

--- INITIATING FINAL RISK CLASSIFICATION ---
Loading True Eco-Triad Radar...
Calculating statistical thresholds across 2,413,579 active events...
  -> Level 2 (Warning) Threshold: 295,583.55
  -> Level 3 (Critical) Threshold: 489,082.93
Classifying spatial-temporal records against the new Eco-Triad scale...
Compressing Final Radar Output...
Pipeline Complete. Triggering download for nalbari_je_radar_final_classified.csv.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SYSTEM SUMMARY (Total Days Across All Zones) ---
alert_level
Level 0 (Safe)         511166
Level 1 (Monitor)     1810184
Level 2 (Warning)      362037
Level 3 (Critical)     241358
Name: count, dtype: int64


In [11]:
!pip install -q folium h3 pandas

import pandas as pd
import h3
import folium
import json
from google.colab import files

print("--- GENERATING GEOSPATIAL RADAR MAP ---")

print("Loading Final Classified Data...")
df = pd.read_csv("nalbari_je_radar_final_classified.csv.gz")

print("Aggregating Chronic Risk Hotspots...")
# Count exactly how many days each hexagon spent at Level 3 Critical
level_3_data = df[df['alert_level'] == 3]
hotspots = level_3_data.groupby('hexagon').size().reset_index(name='level_3_days')

# Merge back to ensure we have all hexagons (even the perfectly safe ones)
all_hexes = pd.DataFrame({'hexagon': df['hexagon'].unique()})
hotspots = pd.merge(all_hexes, hotspots, on='hexagon', how='left').fillna(0)

print("Building Interactive Folium Map Engine...")
# Center map geographically on Nalbari
m = folium.Map(location=[26.44, 91.44], zoom_start=11, tiles="CartoDB positron")

# Generate GeoJSON features for your physical H3 hexagons
features = []
for _, row in hotspots.iterrows():
    hex_id = row['hexagon']
    risk_days = row['level_3_days']

    # Get physical boundary points for the hexagon
    boundary = h3.cell_to_boundary(hex_id)
    # H3 returns (lat, lng), but GeoJSON strictly requires (lng, lat)
    geom = [[ [lng, lat] for lat, lng in boundary ]]

    # The Visual Filter: Color code based on chronic danger
    if risk_days > 150:
        color = "#ff0000" # Severe Red (Constant outbreak zone)
        fill_opacity = 0.6
    elif risk_days > 75:
        color = "#ff6600" # Orange
        fill_opacity = 0.5
    elif risk_days > 0:
        color = "#ffcc00" # Yellow (Occasional risk)
        fill_opacity = 0.4
    else:
        color = "#00ff00" # Green (Mathematically Safe)
        fill_opacity = 0.1

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": geom
        },
        "properties": {
            "hexagon_id": hex_id,
            "days_at_level_3": risk_days,
            "style": {
                "fillColor": color,
                "color": color,
                "weight": 1,
                "fillOpacity": fill_opacity
            }
        }
    }
    features.append(feature)

geojson_data = {"type": "FeatureCollection", "features": features}

# Add the interactive layer to the map
folium.GeoJson(
    geojson_data,
    style_function=lambda x: x['properties']['style'],
    tooltip=folium.GeoJsonTooltip(fields=['hexagon_id', 'days_at_level_3'], aliases=['Vector Zone:', 'Total Days at Level 3:'])
).add_to(m)

map_file = "Nalbari_JEV_Hotspot_Map.html"
print("Rendering map interface...")
m.save(map_file)

print(f"Map successfully generated. Triggering download for {map_file}...")
files.download(map_file)

--- GENERATING GEOSPATIAL RADAR MAP ---
Loading Final Classified Data...
Aggregating Chronic Risk Hotspots...
Building Interactive Folium Map Engine...
Rendering map interface...
Map successfully generated. Triggering download for Nalbari_JEV_Hotspot_Map.html...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>